# HGSOC Precision Oncology: PAT001 Walkthrough

This notebook demonstrates the full multi-modal analysis pipeline for a synthetic
patient (PAT001). No real patient data is used. All tools operate in DRY_RUN mode
unless you configure live MCP connections.

In [ ]:
import sys, os
# Ensure repo root is on the path so the fixture can be imported
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

from tests.fixtures.pat001_canonical import PAT001

print("Canonical values loaded:", PAT001)

## Step 1 — Genomic Instability

**HRD (Homologous Recombination Deficiency)** measures how well cancer cells
repair DNA breaks — a score above 42 indicates significant repair dysfunction.

**TMB (Tumor Mutational Burden)** counts mutations per megabase of DNA — higher
values often correlate with better immunotherapy response.

In [ ]:
# In a live MCP session, this would call mcp-genomic-results.
# Here we verify the canonical values are correct.
hrd_score = PAT001["hrd_score"]
tmb = PAT001["tmb_mut_per_mb"]

print(f"HRD score: {hrd_score}")
print(f"TMB: {tmb} mut/Mb")

assert hrd_score == 72, f"Expected HRD 72, got {hrd_score}"
assert tmb == 4.2, f"Expected TMB 4.2, got {tmb}"
print("Genomic instability values verified.")

## Step 2 — Neoantigen Prediction

Neoantigens are mutant protein fragments that may help the immune system recognize
cancer cells. The system predicts which peptides bind tightly to MHC-I molecules
(lower IC50 = stronger binding = better immune recognition).

In [ ]:
# In a live MCP session, this would call mcp-neoantigen predict_mhc1_binding.
peptide = PAT001["top_neoantigen_peptide"]
ic50 = PAT001["top_neoantigen_ic50_nm"]
hla = PAT001["hla_allele"]

print(f"Peptide: {peptide}")
print(f"HLA allele: {hla}")
print(f"Predicted IC50: {ic50} nM")
print(f"Binding strength: {'Strong' if ic50 < 50 else 'Weak'} binder")

assert ic50 == 7.8, f"Expected IC50 7.8, got {ic50}"
print("Neoantigen prediction verified.")

## Step 3 — Spatial Transcriptomics

Spatial transcriptomics maps gene activity across a tissue slice. Each 'spot' is
a ~50µm region. Moran's I measures spatial clustering: values near 0 indicate
random distribution; positive values indicate clustering.

In [ ]:
# In a live MCP session, this would call mcp-spatialtools.
spots = PAT001["spatial_spot_count"]
morans_i = PAT001["morans_i_global"]

print(f"Spatial spots analyzed: {spots}")
print(f"Global Moran's I: {morans_i}")
print(f"Interpretation: {'Random distribution' if abs(morans_i) < 0.05 else 'Spatial clustering detected'}")

assert spots == 300, f"Expected 300 spots, got {spots}"
print("Spatial transcriptomics values verified.")

## Step 4 — Cell Type Deconvolution

Deconvolution estimates how many cells of each type are present in the tumor
microenvironment.

In [ ]:
# In a live MCP session, this would call mcp-cibersortx or mcp-spatialtools.
deconv = PAT001["deconvolution"]

print("Cell type deconvolution results:")
for cell_type, count in sorted(deconv.items(), key=lambda x: -x[1]):
    print(f"  {cell_type:20s} {count} cells")

assert deconv["cd8_t_cells"] == 30, f"Expected CD8 count 30, got {deconv['cd8_t_cells']}"
print("\nDeconvolution values verified.")

## Step 5 — Perturbation Prediction

The GEARS model predicts how gene knockdowns would change cancer cell behavior.
NNMT is a metabolic enzyme associated with HGSOC drug resistance.

In [ ]:
# In a live MCP session, this would call mcp-perturbation with NNMT+ctrl.
# Here we demonstrate the expected workflow.
print("Perturbation: NNMT knockdown (NNMT+ctrl)")
print("Expected effect: Recovery of immune markers in tumor microenvironment")
print("")
print("Note: Run this cell with a live MCP connection to see predicted")
print("delta expression values from the GEARS GNN model.")

## Putting it all together

High HRD (72) predicts sensitivity to PARP inhibitors. High-affinity neoantigens
(IC50=7.8 nM) suggest the immune system can recognize this tumor. CD8+ T cell
infiltration (30 cells) confirms immune activity. Together, these findings support
a hypothesis of combined PARP inhibition + checkpoint blockade.

> **Teaching note:** This is a synthetic demonstration. Clinical decisions require
> pathologist and oncologist review.

### Discussion questions

1. Why does HRD predict PARP inhibitor sensitivity?
2. What threshold IC50 is considered 'strong' MHC-I binding?
3. How would you interpret Moran's I = -0.0033 biologically?